<a href="https://colab.research.google.com/github/hsb0205/AI-CLASS/blob/main/WEEK15/spam_mail_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# ------------------------------------------------------------
# 1. 훈련 문서와 정답 라벨 만들기
# ------------------------------------------------------------

# docs는 메일 제목 또는 짧은 문장 데이터이다.
docs = [
    "additional income",
    "best price",
    "big bucks",
    "billion",
    "earn extra cash",
    "earn money",
    "spring savings certificate",
    "valero gas marketing",
    "all domestic employees",
    "nominations for oct",
    "confirmation from spinner"
]

# labels는 각 문장의 정답이다.
# 1은 스팸 메일, 0은 정상 메일이다.
labels = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

In [4]:
# ------------------------------------------------------------
# 2. 단어를 정수 인덱스로 변환
# ------------------------------------------------------------

# vocab_size는 사용할 단어 사전의 크기이다.
# 여기서는 50개의 번호 안에서 단어를 정수로 바꾼다.
vocab_size = 50

# one_hot() 함수는 문장에 있는 단어를 정수 인덱스로 바꿔준다.
# 주의: 여기서 one_hot은 실제 원-핫 벡터가 아니라 정수 인코딩에 가깝다.
encoded_docs = [one_hot(d, vocab_size) for d in docs]

print("정수 인코딩 결과")
print(encoded_docs)

정수 인코딩 결과
[[16, 43], [33, 22], [31, 7], [8], [36, 36, 4], [36, 39], [42, 11, 43], [44, 47, 24], [32, 17, 44], [3, 7, 33], [24, 43, 3]]


In [5]:
# ------------------------------------------------------------
# 3. 패딩 처리
# ------------------------------------------------------------

# 문장마다 단어 개수가 다르므로 길이를 맞춰야 한다.
# max_length=4는 모든 문장을 단어 4개 길이로 맞춘다는 뜻이다.
max_length = 4

# padding="post"는 부족한 부분을 문장 뒤쪽에 0으로 채운다는 뜻이다.
padded_docs = pad_sequences(
    encoded_docs,
    maxlen=max_length,
    padding="post"
)

print("\n패딩 결과")
print(padded_docs)


패딩 결과
[[16 43  0  0]
 [33 22  0  0]
 [31  7  0  0]
 [ 8  0  0  0]
 [36 36  4  0]
 [36 39  0  0]
 [42 11 43  0]
 [44 47 24  0]
 [32 17 44  0]
 [ 3  7 33  0]
 [24 43  3  0]]


In [6]:
# ------------------------------------------------------------
# 4. 신경망 모델 만들기
# ------------------------------------------------------------

# Sequential은 층을 순서대로 쌓는 모델이다.
model = Sequential()

# Embedding 층
# 정수 인덱스로 된 단어를 8차원의 실수 벡터로 변환한다.
model.add(Embedding(vocab_size, 8, input_length=max_length))

# Flatten 층
# Embedding 결과를 1차원으로 펼친다.
model.add(Flatten())

# 출력층
# 스팸 또는 정상으로 분류하는 이진 분류이므로 출력 노드는 1개이다.
# sigmoid는 0과 1 사이의 값을 출력한다.
model.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:
# ------------------------------------------------------------
# 5. 모델 컴파일
# ------------------------------------------------------------

# optimizer="adam":
# 가중치를 업데이트하는 최적화 알고리즘이다.
#
# loss="binary_crossentropy":
# 정답이 0 또는 1인 이진 분류에서 사용하는 손실 함수이다.
#
# metrics=["accuracy"]:
# 정확도를 평가 지표로 사용한다.
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\n모델 구조")
model.summary()


모델 구조


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# ------------------------------------------------------------
# 6. 모델 학습
# ------------------------------------------------------------

# epochs=50은 전체 데이터를 50번 반복해서 학습한다는 뜻이다.
# verbose=0은 학습 과정을 화면에 출력하지 않는다는 뜻이다.
model.fit(padded_docs, labels, epochs=50, verbose=0)

In [9]:
# ------------------------------------------------------------
# 7. 모델 평가
# ------------------------------------------------------------

# evaluate()는 손실값과 정확도를 반환한다.
loss, accuracy = model.evaluate(padded_docs, labels, verbose=0)

print("\n정확도=", accuracy)


정확도= 0.9090909361839294


In [10]:
# ------------------------------------------------------------
# 8. 새로운 문서 테스트
# ------------------------------------------------------------

# 학습에 사용하지 않은 새로운 문장을 테스트한다.
test_doc = ["big income"]

# 새로운 문장도 훈련 데이터와 같은 방식으로 정수 인코딩한다.
encoded_test_docs = [one_hot(d, vocab_size) for d in test_doc]

# 새로운 문장도 훈련 데이터와 같은 길이로 패딩한다.
padded_test_docs = pad_sequences(
    encoded_test_docs,
    maxlen=max_length,
    padding="post"
)

# predict()는 스팸일 확률에 가까운 값을 출력한다.
prediction = model.predict(padded_test_docs)

print("\n테스트 문서 예측값")
print(prediction)


# 예측값이 0.5보다 크면 스팸, 아니면 정상으로 판단한다.
if prediction[0][0] > 0.5:
    print("스팸 메일로 예측됩니다.")
else:
    print("정상 메일로 예측됩니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step

테스트 문서 예측값
[[0.58323395]]
스팸 메일로 예측됩니다.
